In [ ]:
# import pandas as pd
# name='352'
# long_name='3524m031'
# band='w1'
# table = pd.read_csv('./mached_catalog/'+name+'/'+long_name+'/'+long_name+'_'+ band +'_mached.csv')

In [ ]:
import numpy as np
import pandas as pd

def calculate_magnitude(flux):
    if flux <= 0:
        return np.nan
    return 22.5 - 2.5 * np.log10(flux)

def calculate_error(flux,dflux):
    mag_upper = calculate_magnitude(flux - dflux)
    mag_lower = calculate_magnitude(flux + dflux)
    dmag = (mag_upper - mag_lower) / 2
    return dmag

cal_mag_ufunc  = np.frompyfunc(calculate_magnitude,1,1)
cal_error_ufunc  = np.frompyfunc(calculate_error,2,1)
def make_single_light_curve(table, index):
    line = table.iloc[index]
    line_len = len(line)
    ra = line[0]
    dec = line[1]
    flux_unfiltered = line[3:line_len:3]
    flux = np.array(flux_unfiltered[flux_unfiltered.notnull()])
    dflux_unfiltered = line[4:line_len:3]
    dflux = np.array(dflux_unfiltered[dflux_unfiltered.notnull()])
    mjdmean_unfiltered = line[5:line_len:3]
    mjdmean = np.array(mjdmean_unfiltered[mjdmean_unfiltered.notnull()])
    assert len(flux)==len(dflux) and len(dflux)==len(mjdmean), 'light curve uncomplete!'
    mag = cal_mag_ufunc(flux)
    error = cal_error_ufunc(flux,dflux)
    return ra, dec, mag, error, mjdmean

In [ ]:
import time
import pickle
def one_footprint_cal(name,long_name,band):
    start = time.time()
    table = pd.read_csv('./mached_catalog/'+name+'/'+long_name+'/'+long_name+'_'+ band +'_mached.csv')
    result_list = []
    for i in range(0,len(table)):
        ra, dec, mags, errors, mjdmean = make_single_light_curve(table,i)
        # if len(mags)>=10:
        #     kai_squre = 0
        #     mean_mag = np.mean(mags)
        #     for mag,error in zip(mags,errors):
        #         kai_squre += ((mag-mean_mag)/error)**2
        #     result_line = np.array([i,ra,dec,mean_mag,kai_squre])
        #     result_list.append(result_line)

        result_list.append([ra, dec, mags, errors, mjdmean])
    fw = open('./mached_catalog/'+name+'/'+long_name+'/'+long_name+'_'+ band +'_light_curves.dat','wb')
    pickle.dump(result_list,fw)

    # result_table = pd.DataFrame(np.array(result_list),columns=['id_in_matched','ra','dec','Mean','kai_squre'])
    # print(result_table)
    # result_table.to_csv('./mached_catalog/'+name+'/'+long_name+'/'+long_name+'_'+ band +'kai_square.csv')
    end = time.time()
    print('Task %s runs %0.2f seconds.' % (long_name, (end - start)))

In [ ]:
one_footprint_cal('000','0000m016','w1')

In [ ]:
from multiprocessing import Pool
import time
import glob
import os

thread_pool = Pool()
first_layer_names = glob.glob('[0-9][0-9][0-9]',root_dir='./mached_catalog')
for name in first_layer_names:
        second_layer_names = glob.glob('[0-9][0-9][0-9][0-9]*',
                                    root_dir='./mached_catalog/'+name)
        for long_name in second_layer_names:
            for band in ('w1', 'w2'):
                if True == os.path.isfile('./mached_catalog/'+name+'/'+long_name+'/'+long_name+'_'+ band +'_mached.csv'):
                    thread_pool.apply_async(one_footprint_cal,args=(name,long_name,band))
print('Waiting for all subprocesses done...')
thread_pool.close()
thread_pool.join()
print('all subprocesses done')

In [ ]:
# import matplotlib.pyplot as plt
# plt.hist(np.array(result_table['kai_squre']),bins=100,range=(0,100))